# Kaggriculture Immitation Deep Reinforcement Replay Controller

## Dueling Double DQN with Action Branching for the Kaggriculture simulation.

This module provides the core RL components for training an off-policy agent
that navigates a multi-branch action space:
    - Farmer actions    (15 discrete actions)
    - Hand actions      (6 hands × 15 discrete actions each)
    - Market actions    (10 discrete actions)

Rather than a flat action space of 15 × 15^6 × 10 ≈ 2.9 × 10^12, the network
uses **action branching** with 122 Q-value outputs, combined through a
**Dueling** architecture that separates state value V(s) from action advantage
A(s, a) to stabilize learning.

The bootstrap is replay buffer trained on the daily top agent actions.

| Root | Writable | Purpose |
|------|----------|---------|
| `/kaggle/input/` | **No** | All attached datasets (read-only). Episode JSONs: `/kaggle/input/kaggriculture-episodes-YYYY-MM-DD/{id}.json`. Code dataset: `/kaggle/input/datasets/scottweeden/self-training-code/`. |
| `/kaggle/working/` | **Yes** | Starts empty. Writable training modules, metadata cache, checkpoints, metrics, `agent.py`. Adapter modules stay on **input** (`self-training-code`). |

Bootstrap reads episodes from **input** mounts (date range **2026-07-30 → 2026-08-27**); metadata and training artifacts go to **working**.

**Training mode:** defaults to `"dry_run"` locally (~1 min smoke test). Set env `KAGGLE_TRAINING_MODE=medium` on Kaggle GPU for production (~2–3 h per 5-day batch). Progress persists in `run/metrics/bootstrap_state.json` and is restored from the code dataset on re-run.

**GPU:** Settings → Accelerator → GPU T4 x2 → Save Version → Run All.


## 1. Setup — deploy code, merge metadata, configure training

In [6]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# ── /kaggle/input (read-only) ─────────────────────────────────────────────
KAGGLE_INPUT = (
    Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("~/kagg").expanduser()
).resolve()
CODE_CANDIDATES = [
    KAGGLE_INPUT / "datasets" / "scottweeden" / "self-training-code",
    KAGGLE_INPUT / "self-training-code",
    Path("/kaggle/input/datasets/scottweeden/kaggriculture-self-training-code"),
    Path("/kaggle/input/kaggriculture-self-training-code"),
]
CODE_SRC = next((p for p in CODE_CANDIDATES if (p / "episode_catalog.py").exists()), None)

# ── /kaggle/working (writable; starts empty) ──────────────────────────────
KAGGLE_WORKING = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("~/kagg/working").expanduser()
).resolve()
METADATA_DIR = KAGGLE_WORKING / "kaggle_episodes"   # metadata.json + daily_manifests/
METADATA_PATH = METADATA_DIR / "metadata.json"
EXPERIMENT_DIR = KAGGLE_WORKING / "run"             # checkpoints/, metrics/, agent.py (not code root)

READ_ONLY_MODULES = {
    "kaggriculture_adapter.py",
    "kaggriculture_path_b_rebuild.py",
}
TRAINING_MODULES = [
    "kaggriculture_self_play_training.py",
    "kaggriculture_dataset_publish.py",
    "episode_catalog.py",
    "path_b_bootstrap.py",
    "kaggriculture_adapter.py",
    "kaggriculture_path_b_rebuild.py",
    "kaggle_env_wrapper.py",
    "dataset_loader.py",
    "eval_policy.py",
    "visualize.py",
]
WRITABLE_MODULES = [m for m in TRAINING_MODULES if m not in READ_ONLY_MODULES]

if CODE_SRC is None:
    raise FileNotFoundError(
        "Missing code dataset. Tried: "
        + ", ".join(str(p) for p in CODE_CANDIDATES)
        + ". Attach scottweeden/self-training-code."
    )
for name in READ_ONLY_MODULES:
    if not (CODE_SRC / name).exists():
        raise FileNotFoundError(f"Missing read-only module in code dataset: {CODE_SRC / name}")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
for name in WRITABLE_MODULES:
    src = CODE_SRC / name
    if not src.exists():
        raise FileNotFoundError(f"Missing {src}")
    shutil.copy2(src, KAGGLE_WORKING / name)
shutil.copytree(CODE_SRC / "kaggriculture_rl", KAGGLE_WORKING / "kaggriculture_rl", dirs_exist_ok=True)

print(f"Code (input, read-only adapter): {CODE_SRC}")
print(f"Adapter modules imported from input; training modules copied to: {KAGGLE_WORKING}")

os.chdir(KAGGLE_WORKING)
sys.path.insert(0, str(KAGGLE_WORKING))
sys.path.insert(0, str(CODE_SRC))  # adapter/path_b_rebuild from input dataset

# Kernel reruns cache old modules — drop stale copies after copying fresh code.
for _mod in (
    "episode_catalog",
    "kaggriculture_adapter",
    "path_b_bootstrap",
    "kaggriculture_dataset_publish",
    "path_b_bootstrap",
    "kaggriculture_path_b_rebuild",
    "kaggle_env_wrapper",
    "dataset_loader",
    "eval_policy",
    "visualize",
):
    sys.modules.pop(_mod, None)

from episode_catalog import (
    DEFAULT_END_DATE,
    DEFAULT_START_DATE,
    configure_local_datasets_root,
    ensure_daily_episode_dataset,
    ensure_episode_datasets_for_range,
    ensure_kaggle_index_dataset,
    is_kaggle_runtime,
    kaggle_daily_dataset_dir,
    merge_episode_metadata,
    pick_next_bootstrap_days,
    resolve_episode_paths_from_metadata,
    save_metadata,
)
from path_b_bootstrap import (
    bootstrap_metadata_start_date,
    load_bootstrap_state,
    merge_bootstrap_state_from_code_dataset,
    plan_next_bootstrap_days_from_state,
)
from kaggriculture_adapter import any_accelerator_available, gpu_backend_diagnostics
from kaggriculture_self_play_training import train_self_play

# Default dry_run locally (~1 min). Override with KAGGLE_TRAINING_MODE=medium|full on Kaggle GPU.
TRAINING_MODE = os.environ.get("KAGGLE_TRAINING_MODE", "dry_run")
if TRAINING_MODE not in ("dry_run", "medium", "full"):
    raise ValueError(
        f"KAGGLE_TRAINING_MODE={TRAINING_MODE!r} invalid; use dry_run, medium, or full"
    )

MODE_PRESETS = {
    "dry_run": {
        "bootstrap_mode": "daily_incremental",
        "bootstrap_days_per_run": 1,
        "bootstrap_episodes": None,
        "bootstrap_top_per_day": None,
        "bootstrap_passes": 1,
        "bootstrap_transitions": None,
        "buffer_capacity": 10_000,
        "bc_epochs_per_pass": 1,
        "bc_steps_per_epoch": 50,
        "total_episodes": 2,
        "learning_start_episodes": 99,
        "n_eval_episodes": 0,
        "max_episode_steps": 50,
        "_eta": "~1 minute",
    },
    "medium": {
        "bootstrap_mode": "daily_incremental",
        "bootstrap_days_per_run": 5,
        "bootstrap_episodes": None,
        "bootstrap_top_per_day": None,
        "bootstrap_passes": 1,
        "bootstrap_transitions": None,
        "buffer_capacity": 200_000,
        "bc_epochs_per_pass": 1,
        "bc_steps_per_epoch": None,
        "total_episodes": 25,
        "learning_start_episodes": 5,
        "n_eval_episodes": 10,
        "max_episode_steps": 720,
        "_eta": "~2–3 hours per 5-day batch",
    },
    "full": {
        "bootstrap_mode": "daily_incremental",
        "bootstrap_days_per_run": 5,
        "bootstrap_episodes": None,
        "bootstrap_top_per_day": None,
        "bootstrap_passes": 1,
        "bootstrap_transitions": None,
        "buffer_capacity": 600_000,
        "bc_epochs_per_pass": 2,
        "bc_steps_per_epoch": None,
        "total_episodes": 100,
        "learning_start_episodes": 5,
        "n_eval_episodes": 20,
        "max_episode_steps": 720,
        "_eta": "several hours per 5-day batch",
    },
}

if TRAINING_MODE not in MODE_PRESETS:
    raise ValueError(
        f"Unknown TRAINING_MODE={TRAINING_MODE!r}; choose from {list(MODE_PRESETS)}"
    )

_mode = MODE_PRESETS[TRAINING_MODE]
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

from kaggriculture_dataset_publish import restore_training_artifacts_from_code_dataset

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
_restored = restore_training_artifacts_from_code_dataset(CODE_SRC, EXPERIMENT_DIR)
_bootstrap_state = merge_bootstrap_state_from_code_dataset(CODE_SRC, EXPERIMENT_DIR)
if _restored:
    print(f"Restored from code dataset training_artifacts/: {_restored}")
_done = list(_bootstrap_state.get("bootstrapped_dates", []))
if _done:
    print(f"Bootstrap resume: {len(_done)} day(s) already complete through {_done[-1]}")

_resume_ckpt = EXPERIMENT_DIR / "checkpoints" / "training_state_latest.pt"
_bootstrap_state_path = EXPERIMENT_DIR / "metrics" / "bootstrap_state.json"

TRAINING_CONFIG = {
    "experiment_dir": str(EXPERIMENT_DIR),
    "code_src": str(CODE_SRC),
    "use_kaggle_env": True,
    "bootstrap_mode": _mode["bootstrap_mode"],
    "bootstrap_days_per_run": _mode["bootstrap_days_per_run"],
    "bootstrap_episodes": _mode["bootstrap_episodes"],
    "bootstrap_top_per_day": _mode["bootstrap_top_per_day"],
    "bootstrap_passes": _mode["bootstrap_passes"],
    "bootstrap_transitions": _mode["bootstrap_transitions"],
    "buffer_capacity": _mode["buffer_capacity"],
    "bc_epochs": 0,
    "bc_epochs_per_pass": _mode["bc_epochs_per_pass"],
    "bc_steps_per_epoch": _mode["bc_steps_per_epoch"],
    "data_dir": str(METADATA_DIR),
    "metadata_path": str(METADATA_PATH),
    "download_bootstrap": False,
    "bc_batch_size": 64,
    "total_episodes": _mode["total_episodes"],
    "learning_start_episodes": _mode["learning_start_episodes"],
    "batch_size": 32,
    "checkpoint_interval": 10,
    "n_eval_episodes": _mode["n_eval_episodes"],
    "max_episode_steps": _mode["max_episode_steps"],
    "device_name": "auto",
    "seed": 42,
    "resume": str(EXPERIMENT_DIR) if (_resume_ckpt.exists() or _restored) else None,
    "publish_code_dataset": TRAINING_MODE != "dry_run",
    "verbose": True,
}

print(f"=== TRAINING_MODE={TRAINING_MODE!r} ({_mode['_eta']}) ===")
if _mode["bootstrap_mode"] == "daily_incremental":
    print(
        f"  Daily bootstrap: {_mode['bootstrap_days_per_run']} chronological unseen day(s)/run, "
        f"all episodes/day, bc_epochs/day={_mode['bc_epochs_per_pass']}, "
        f"bc_steps={_mode['bc_steps_per_epoch'] or 'all'}"
    )
    if _done:
        print(f"  Bootstrapped days ({len(_done)}): {_done}")
    if TRAINING_CONFIG["resume"]:
        print(f"  Resume checkpoint: {_resume_ckpt}")
else:
    print(
        f"  Bootstrap: {_mode['bootstrap_passes']} passes × {_mode['bootstrap_transitions']:,} trans/pass "
        f"(buffer {_mode['buffer_capacity']:,})"
    )
print(
    f"  Self-play: {_mode['total_episodes']} episodes "
    f"(learn from ep {_mode['learning_start_episodes']}, "
    f"max_steps={_mode['max_episode_steps']}, eval={_mode['n_eval_episodes']})"
)

# GPU diagnostics — CUDA (Kaggle), MLX/MPS (Apple Silicon), or CPU fallback
print("=== GPU diagnostics ===")
print(f"torch: {torch.__version__} (+cu{'yes' if '+cu' in torch.__version__ else '?'})")
_gpu = gpu_backend_diagnostics()
print(f"torch.cuda.is_available(): {_gpu['cuda_available']}")
print(f"torch.backends.mps.is_available(): {_gpu['mps_available']}")
print(f"mlx.metal.is_available(): {_gpu['mlx_available']}")
if _gpu.get("cuda_device"):
    print(f"CUDA device: {_gpu['cuda_device']}")
if _gpu.get("mlx_default_device"):
    print(f"MLX default device: {_gpu['mlx_default_device']}")
print(f"Resolved training device: {_gpu['resolved_device']}")
print(f"KAGGLE_KERNEL_RUN_TYPE: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '(unset)')}")
print(f"NVIDIA_VISIBLE_DEVICES: {os.environ.get('NVIDIA_VISIBLE_DEVICES', '(unset)')}")
try:
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True, stderr=subprocess.STDOUT).strip())
except Exception as exc:
    print(f"nvidia-smi: {exc}")

if any_accelerator_available():
    if _gpu["cuda_available"]:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    elif _gpu["mps_available"]:
        print("Apple GPU: PyTorch MPS backend")
    elif _gpu["mlx_available"]:
        print("Apple GPU: MLX Metal backend (PyTorch training uses CPU unless MPS is enabled)")
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    raise RuntimeError(
        "No GPU in this Batch session (31 episode datasets attached). "
        "CLI push often schedules CPU-only workers for large multi-dataset kernels. "
        "Fix: open notebook on Kaggle → Settings → Accelerator → GPU T4 x2 → Save Version → Run All."
    )
else:
    print("WARNING: No CUDA/MLX/MPS accelerator detected; training will use CPU.")

# Preflight: input mounts exist, working is writable
assert KAGGLE_INPUT.is_dir(), f"Missing {KAGGLE_INPUT}"
assert os.access(KAGGLE_WORKING, os.W_OK), f"Not writable: {KAGGLE_WORKING}"

LOCAL_DATASETS = KAGGLE_INPUT / "datasets" / "kaggle"
_next_days = plan_next_bootstrap_days_from_state(
    _done,
    n_days=_mode["bootstrap_days_per_run"],
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
)
if not is_kaggle_runtime():
    configure_local_datasets_root(LOCAL_DATASETS)
    ensure_kaggle_index_dataset(local_root=LOCAL_DATASETS)
    for _day in _next_days:
        ensure_daily_episode_dataset(_day, local_root=LOCAL_DATASETS)
    example_episode_dir = (
        kaggle_daily_dataset_dir(_next_days[0])
        if _next_days
        else kaggle_daily_dataset_dir(DEFAULT_END_DATE)
    )
    print(f"Local episode datasets: {LOCAL_DATASETS}")
    print(f"Next bootstrap day(s): {_next_days or '(corpus complete)'}")
else:
    example_episode_dir = kaggle_daily_dataset_dir(
        _next_days[0] if _next_days else DEFAULT_END_DATE
    )
    if not example_episode_dir.is_dir():
        raise FileNotFoundError(f"Missing episode mount: {example_episode_dir}")

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_DIR.mkdir(parents=True, exist_ok=True)
_metadata_start = bootstrap_metadata_start_date(_done, default_start=DEFAULT_START_DATE)
print(f"Metadata index from {_metadata_start} → {DEFAULT_END_DATE} (skipping {len(_done)} bootstrapped day(s))")
metadata = merge_episode_metadata(
    data_dir=METADATA_DIR,
    start_date=_metadata_start,
    end_date=DEFAULT_END_DATE,
)
save_metadata(metadata, METADATA_PATH)

print(f"Code (input): {CODE_SRC}")
print(f"Episode mount example (input): {example_episode_dir}")
print(f"Metadata cache (working): {METADATA_DIR}")
print(f"Experiment output (working): {EXPERIMENT_DIR}")
print(f"Episodes indexed: {metadata['total_episodes_indexed']:,}")

_cfg_bc_steps = TRAINING_CONFIG.get("bc_steps_per_epoch") or "all"
if _mode["bootstrap_mode"] == "daily_incremental":
    _next = plan_next_bootstrap_days_from_state(
        _done,
        n_days=_mode["bootstrap_days_per_run"],
        start_date=DEFAULT_START_DATE,
        end_date=DEFAULT_END_DATE,
    )
    print(
        f"Bootstrap plan: daily_incremental, {_mode['bootstrap_days_per_run']} day(s)/run, "
        f"bc_epochs/day={_mode['bc_epochs_per_pass']}, bc_steps={_cfg_bc_steps}, "
        f"training_mode={TRAINING_MODE!r}"
    )
    print(f"  Already bootstrapped ({len(_done)}): {_done or '(none)'}")
    print(f"  Next days this run: {_next or '(corpus complete)'}")
else:
    episode_files = resolve_episode_paths_from_metadata(
        metadata,
        top_per_day=TRAINING_CONFIG["bootstrap_top_per_day"],
        max_episodes=TRAINING_CONFIG["bootstrap_episodes"],
    )
    _cfg_passes = TRAINING_CONFIG["bootstrap_passes"]
    _cfg_trans = TRAINING_CONFIG.get("bootstrap_transitions") or TRAINING_CONFIG["buffer_capacity"]
    print(f"Bootstrap episodes resolved: {len(episode_files):,}")
    print(
        f"Bootstrap plan: {_cfg_passes} passes × {_cfg_trans:,} trans/pass, "
        f"bc_steps_per_epoch={_cfg_bc_steps}, training_mode={TRAINING_MODE!r}"
    )
    if episode_files:
        print(f"  Sample: {episode_files[0]}")

print(f"Replay buffer capacity: {TRAINING_CONFIG['buffer_capacity']:,}")
print(json.dumps(TRAINING_CONFIG, indent=2))


Code (input, read-only adapter): /Users/sweeden/kagg/datasets/scottweeden/self-training-code
Adapter modules imported from input; training modules copied to: /Users/sweeden/kagg/working
Successfully imported Kaggriculture Path B components.
Restored from code dataset training_artifacts/: ['models/model.pth', 'checkpoints/training_state_latest.pt', 'metrics/bootstrap_state.json', 'metrics/bc_pretrain.json', 'metrics/episode_metrics.json', 'metrics/win_rate_eval.json', 'config.json', 'agent.py', 'plots/']
=== TRAINING_MODE='dry_run' (~1 minute) ===
  Daily bootstrap: 1 chronological unseen day(s)/run, all episodes/day, bc_epochs/day=1, bc_steps=50
  Bootstrapped days (16): ['2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05', '2026-08-06', '2026-08-07', '2026-08-08', '2026-08-09', '2026-08-10', '2026-08-11', '2026-08-12', '2026-08-13', '2026-08-14']
  Resume checkpoint: /Users/sweeden/kagg/working/run/checkpoints/training_state_latest.pt
  Sel

100%|██████████| 69.1k/69.1k [00:00<00:00, 618kB/s]


Dataset URL: https://www.kaggle.com/datasets/kaggle/kaggriculture-episodes-2026-07-31
License(s): CC0-1.0


100%|██████████| 76.2k/76.2k [00:00<00:00, 187kB/s]



Dataset URL: https://www.kaggle.com/datasets/kaggle/kaggriculture-episodes-2026-08-01
License(s): CC0-1.0



...

Dataset URL: https://www.kaggle.com/datasets/kaggle/kaggriculture-episodes-2026-08-25
License(s): CC0-1.0



100%|██████████| 56.5k/56.5k [00:00<00:00, 1.47MB/s]


Dataset URL: https://www.kaggle.com/datasets/kaggle/kaggriculture-episodes-2026-08-26
License(s): CC0-1.0



100%|██████████| 56.8k/56.8k [00:00<00:00, 521kB/s]


Code (input): /Users/sweeden/kagg/datasets/scottweeden/self-training-code
Episode mount example (input): /Users/sweeden/kagg/datasets/kaggle/kaggriculture-episodes-2026-08-27
Metadata cache (working): /Users/sweeden/kagg/working/kaggle_episodes
Experiment output (working): /Users/sweeden/kagg/working/run
Episodes indexed: 20,879
Bootstrap plan: daily_incremental, 1 day(s)/run, bc_epochs/day=1, bc_steps=50, training_mode='dry_run'
  Already bootstrapped (16): ['2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05', '2026-08-06', '2026-08-07', '2026-08-08', '2026-08-09', '2026-08-10', '2026-08-11', '2026-08-12', '2026-08-13', '2026-08-14']
  Next days this run: ['2026-08-15']
Replay buffer capacity: 10,000
{
  "experiment_dir": "/Users/sweeden/kagg/working/run",
  "code_src": "/Users/sweeden/kagg/datasets/scottweeden/self-training-code",
  "use_kaggle_env": true,
  "bootstrap_mode": "daily_incremental",
  "bootstrap_days_per_run": 1,
  "bootstrap

## 2. Run training (bootstrap → BC → self-play)

In [ ]:
train_self_play(**TRAINING_CONFIG)

## 3. Results summary

In [ ]:
metrics_dir = EXPERIMENT_DIR / "metrics"

bc_path = metrics_dir / "bc_pretrain.json"
if bc_path.exists():
    bc = json.loads(bc_path.read_text())
    print("=== BC Pretrain ===")
    print(f"  Bootstrap transitions: {bc.get('bootstrap_transitions')}")
    print(f"  Final BC loss: {bc.get('final_loss'):.5f}")

wr_path = metrics_dir / "win_rate_eval.json"
if wr_path.exists():
    wr = json.loads(wr_path.read_text())
    print("\n=== Win-rate eval ===")
    print(f"  Win: {wr.get('win_rate', 0):.1%} ({wr.get('wins')}/{wr.get('n_episodes')})")

for rel in ["agent.py", "models/model.pth", "checkpoints/training_state_latest.pt", "config.json"]:
    p = EXPERIMENT_DIR / rel
    print(f"  {'OK' if p.exists() else '--'} {rel}")

## 4. Visualize metrics

In [ ]:
%matplotlib inline

import json
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Re-bootstrap if kernel restarted after long training (setup cell state lost)
KAGGLE_INPUT = (
    Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("~/kagg").expanduser()
).resolve()
KAGGLE_WORKING = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("~/kagg/working").expanduser()
).resolve()
CODE_CANDIDATES = [
    KAGGLE_INPUT / "datasets" / "scottweeden" / "self-training-code",
    KAGGLE_INPUT / "self-training-code",
    Path("/kaggle/input/datasets/scottweeden/kaggriculture-self-training-code"),
    Path("/kaggle/input/kaggriculture-self-training-code"),
]
CODE_SRC = next((p for p in CODE_CANDIDATES if (p / "episode_catalog.py").exists()), None)
for p in (KAGGLE_WORKING, CODE_SRC):
    if p is not None and str(p) not in sys.path:
        sys.path.insert(0, str(p))
if CODE_SRC and not (KAGGLE_WORKING / "visualize.py").exists():
    shutil.copy2(CODE_SRC / "visualize.py", KAGGLE_WORKING / "visualize.py")

EXPERIMENT_DIR = globals().get("EXPERIMENT_DIR", KAGGLE_WORKING / "run")

from visualize import TrainingMetricsLoader, MetricsVisualizer

PLOT_DIR = EXPERIMENT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
metrics_dir = EXPERIMENT_DIR / "metrics"

loader = TrainingMetricsLoader([str(EXPERIMENT_DIR)])
loader.load_all()

if loader.metrics:
    viz = MetricsVisualizer(output_dir=PLOT_DIR, figure_size=(12, 8))
    viz.print_summary(loader)
    figures = viz.plot_comparison(loader)
    if figures:
        exp_names = list(loader.metrics.keys())
        viz.save_figures(figures, exp_names, close=False)
        print(f"Saved {len(figures)} plot(s) to {PLOT_DIR}")
        for fig in figures:
            plt.figure(fig.number)
            plt.tight_layout()
            plt.show()
    else:
        print("No Path B metrics to plot yet — run training first.")
else:
    print(f"No metrics under {EXPERIMENT_DIR}")

bc_path = metrics_dir / "bc_pretrain.json"
if bc_path.exists():
    bc = json.loads(bc_path.read_text())
    losses = bc.get("epoch_losses") or bc.get("stream_stats", {}).get("epoch_losses", [])
    if losses:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(range(1, len(losses) + 1), losses, marker="o")
        ax.set(xlabel="BC epoch / day", ylabel="Loss", title="BC pretrain")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        bc_plot = PLOT_DIR / "bc_pretrain_loss.png"
        fig.savefig(bc_plot, dpi=150, bbox_inches="tight")
        print(f"Saved BC plot to {bc_plot}")
        plt.show()

## 5. Publish artifacts to code dataset

In [ ]:
import os
import shutil
import sys
from pathlib import Path

KAGGLE_INPUT = (
    Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("~/kagg").expanduser()
).resolve()
KAGGLE_WORKING = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("~/kagg/working").expanduser()
).resolve()
CODE_CANDIDATES = [
    KAGGLE_INPUT / "datasets" / "scottweeden" / "self-training-code",
    KAGGLE_INPUT / "self-training-code",
    Path("/kaggle/input/datasets/scottweeden/kaggriculture-self-training-code"),
    Path("/kaggle/input/kaggriculture-self-training-code"),
]
CODE_SRC = next((p for p in CODE_CANDIDATES if (p / "episode_catalog.py").exists()), None)
for p in (KAGGLE_WORKING, CODE_SRC):
    if p is not None and str(p) not in sys.path:
        sys.path.insert(0, str(p))
if CODE_SRC and not (KAGGLE_WORKING / "kaggriculture_dataset_publish.py").exists():
    shutil.copy2(CODE_SRC / "kaggriculture_dataset_publish.py", KAGGLE_WORKING / "kaggriculture_dataset_publish.py")

EXPERIMENT_DIR = globals().get("EXPERIMENT_DIR", KAGGLE_WORKING / "run")
TRAINING_MODE = globals().get("TRAINING_MODE", os.environ.get("KAGGLE_TRAINING_MODE", "dry_run"))

from kaggriculture_dataset_publish import (
    publish_training_artifacts_to_code_dataset,
    training_version_message,
)

if TRAINING_MODE == "dry_run":
    print("Skipping code dataset publish (dry_run mode)")
else:
    summary = publish_training_artifacts_to_code_dataset(
        EXPERIMENT_DIR,
        version_message=training_version_message(EXPERIMENT_DIR),
    )
    print("Published to scottweeden/self-training-code")
    print(f"  Version: {summary['version_message']}")
    print("  Copied:")
    for item in summary["copied_artifacts"]:
        print(f"    - {item}")